# Global MVHI-D downloads — date range

Download original global dekadal MVHI-D GeoTIFFs for cropland (LC-C), GS1 and GS2. No clipping, reprojection or averaging.

Set `START_DATE` and `END_DATE` in YYYY-MM-DD format within the selected `YEAR`. Defaults: 2024-01-01 through 2024-12-31, up to 72 files. Date selection uses each dekad's start date (day 1, 11 or 21), together with the STAC temporal filter.

Set `DROPBOX_DIR` to your existing local Dropbox sync folder. All rasters and logs are saved directly in `FAO/MVHI-D/2024`, with no monthly subfolders. The Dropbox desktop app handles cloud uploads; this notebook does not verify synchronization.

Before running, authenticate in PowerShell with `gcloud auth application-default login`, using an account authorized to read the source bucket. The notebook explicitly loads these credentials and tests source access before downloading. If access fails, the download loop does not start.

Run all cells in order. Existing readable files are skipped. Partial downloads restart on rerun. Missing catalogue entries and failures are reported. The progress bar counts processed files, including failures; the final summary reports successful downloads separately.

In [ ]:
%pip install pystac-client gcsfs requests rasterio tqdm

In [7]:
from pathlib import Path
from urllib.parse import urlparse, unquote
import csv
import os
import re
import time

import google.auth
import gcsfs
import requests
import rasterio
from pystac_client import Client
from tqdm import tqdm

YEAR = 2024
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"
API_ROOT = "https://data.apps.fao.org/geospatial/search/stac"
COLLECTION_ID = "fao-gismgr:ASIS:raster:mapsets:MVHI-D"
SEASONS = ("GS1", "GS2")

# EDIT this to your EXISTING Dropbox sync folder.
# Examples: Path(r"D:\Dropbox"), Path(r"C:\Users\your_name\Dropbox")
#DROPBOX_DIR = Path.home() / "Dropbox"
DROPBOX_DIR = Path(r"C:\Users\DEV-Hao\Dropbox")
OUTPUT_DIR = DROPBOX_DIR / "FAO" / "MVHI-D" / str(YEAR)

# Authenticate in PowerShell first: gcloud auth application-default login
OVERWRITE = False
MAX_ATTEMPTS = 3

from datetime import date

start_date = date.fromisoformat(START_DATE)
end_date = date.fromisoformat(END_DATE)
if start_date > end_date or start_date.year != YEAR or end_date.year != YEAR:
    raise ValueError("Dates must be ordered and within YEAR.")

if not DROPBOX_DIR.is_dir():
    raise FileNotFoundError(
        f"Set DROPBOX_DIR to your existing local Dropbox folder: {DROPBOX_DIR}"
    )
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Save to:", OUTPUT_DIR.resolve())

Save to: C:\Users\DEV-Hao\Dropbox\FAO\MVHI-D\2024


In [2]:
# No bbox or geometry filter: download the original global assets.
client = Client.open(API_ROOT)
items = list(client.search(
    collections=[COLLECTION_ID],
    datetime=f"{START_DATE}T00:00:00Z/{END_DATE}T23:59:59Z",
    limit=100,
).items())  # Follows all result pages.

patterns = [
    re.compile(r"^ASIS-MVHI-D-(\d{4})-(0[1-9]|1[0-2])-D([123])-(GS[12])-LC-C$"),
    re.compile(r"^ASIS\.MVHI-D\.(\d{4})-(0[1-9]|1[0-2])-D([123])\.(GS[12])\.LC-C$"),
]
selected = {}
for item in items:
    short_id = item.id.rsplit(":items:", 1)[-1]
    match = next((m for p in patterns if (m := p.fullmatch(short_id))), None)
    if match is None:
        continue
    year, month, dekad, season = match.groups()
    dekad_start = date(int(year), int(month), (int(dekad) - 1) * 10 + 1)
    if not (start_date <= dekad_start <= end_date) or season not in SEASONS:
        continue
    key = (int(month), int(dekad), season)
    if key in selected:
        if selected[key].id == item.id:
            continue
        raise RuntimeError(f"Duplicate dekad/season: {selected[key].id}, {item.id}")
    selected[key] = item

if not selected:
    raise RuntimeError(f"No matching LC-C items in the selected date range. Example IDs: {[i.id for i in items[:5]]}")

expected = {
    (m, d, s)
    for m in range(1, 13)
    for d in (1, 2, 3)
    for s in SEASONS
    if start_date <= date(YEAR, m, (d - 1) * 10 + 1) <= end_date
}
missing = sorted(expected - selected.keys())
print(f"Found {len(selected)} / {len(expected)} expected LC-C items.")
for month, dekad, season in missing:
    print(f"MISSING: {YEAR}-{month:02d}-D{dekad}-{season}-LC-C")
with (OUTPUT_DIR / f"missing_dekads_{START_DATE}_{END_DATE}.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["year", "month", "dekad", "season", "land_cover"])
    writer.writerows((YEAR, m, d, s, "LC-C") for m, d, s in missing)

Found 72 / 72 expected LC-C items.


In [4]:
session = requests.Session()

def gcs_object(href):
    if href.startswith("gs://"):
        return href[5:]
    parsed = urlparse(href)
    if parsed.hostname in {"storage.googleapis.com", "storage.cloud.google.com"}:
        return unquote(parsed.path.lstrip("/"))
    if (parsed.hostname or "").endswith(".storage.googleapis.com"):
        bucket = parsed.hostname.removesuffix(".storage.googleapis.com")
        return bucket + "/" + unquote(parsed.path.lstrip("/"))
    return None

def choose_source(item):
    # Only consider data/raster assets, never previews or metadata.
    candidates = []
    for key, asset in item.assets.items():
        href = asset.href or ""
        media = (asset.media_type or "").lower()
        roles = asset.roles or []
        if key == "data" or ("data" in roles) or (
            "tiff" in media or urlparse(href).path.lower().endswith((".tif", ".tiff"))
        ):
            if "thumbnail" not in roles and "overview" not in roles:
                candidates.append((key, asset))
    candidates.sort(key=lambda pair: pair[0] != "data")
    if not candidates:
        raise ValueError(f"No raster data asset: {item.id}")
    # Follow the source notebook's preference for GCS alternate links.
    for _, asset in candidates:
        alternate = asset.extra_fields.get("alternate", {}).get("gs")
        href = alternate.get("href") if isinstance(alternate, dict) else alternate
        if href and gcs_object(href):
            return "gcs", gcs_object(href)
        obj = gcs_object(asset.href or "")
        if obj:
            return "gcs", obj
        if (asset.href or "").startswith(("https://", "http://")):
            return "http", asset.href
    raise ValueError(f"No usable source URL: {item.id}")

def validate_tiff(path):
    with rasterio.open(path) as src:
        if src.count < 1 or src.width < 1 or src.height < 1 or src.crs is None:
            raise ValueError(f"Invalid raster: {path}")
        # Header/readability check; does not load the global array into memory.
        src.read(1, window=((0, 1), (0, 1)))

def download(item, destination):
    kind, source = choose_source(item)
    if destination.exists() and not OVERWRITE:
        validate_tiff(destination)
        return "skipped"
    partial = destination.with_suffix(".tif.part")
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            if kind == "gcs":
                expected_size = int(gcs.info(source)["size"])
                gcs.get_file(source, str(partial))
            else:
                with session.get(source, stream=True, timeout=(30, 180),
                                 headers={"Accept-Encoding": "identity"}) as response:
                    response.raise_for_status()
                    size = response.headers.get("Content-Length")
                    expected_size = int(size) if size else None
                    with partial.open("wb") as target:
                        for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                            if chunk:
                                target.write(chunk)
            if partial.stat().st_size == 0:
                raise IOError("Empty download")
            if expected_size is not None and partial.stat().st_size != expected_size:
                raise IOError("Downloaded size differs from source size")
            validate_tiff(partial)
            os.replace(partial, destination)
            return "downloaded"
        except Exception:
            partial.unlink(missing_ok=True)
            if attempt == MAX_ATTEMPTS:
                raise
            time.sleep(2 ** attempt)


In [5]:
# Explicitly load Application Default Credentials; never use anonymous access.
credentials, project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/devstorage.read_only"]
)
gcs = gcsfs.GCSFileSystem(
    token=credentials,
    project=project,
    skip_instance_cache=True,
)

first_item = selected[sorted(selected)[0]]
kind, source = choose_source(first_item)
try:
    if kind == "gcs":
        with gcs.open(source, "rb") as stream:
            header = stream.read(16)
    else:
        with session.get(source, stream=True, timeout=(30, 60)) as response:
            response.raise_for_status()
            header = next(response.iter_content(chunk_size=16), b"")
    if not header:
        raise IOError("The source returned no data.")
except Exception as error:
    raise RuntimeError(
        "Source access test failed. Check Application Default Credentials and "
        "bucket read permissions before starting downloads."
    ) from error
print("Source access test successful.")

Source access test successful.


In [ ]:
manifest = OUTPUT_DIR / f"download_manifest_{START_DATE}_{END_DATE}.csv"
fields = ["item_id", "filename", "status", "bytes", "error"]
rows = []
for (month, dekad, season), item in tqdm(sorted(selected.items()), desc="Global MVHI-D files"):
    filename = f"ASIS-MVHI-D-{YEAR}-{month:02d}-D{dekad}-{season}-LC-C.tif"
    destination = OUTPUT_DIR / filename
    try:
        status = download(item, destination)
        row = dict(item_id=item.id, filename=filename, status=status,
                   bytes=destination.stat().st_size, error="")
    except Exception as error:
        row = dict(item_id=item.id, filename=filename, status="failed", bytes=0, error=str(error))
        tqdm.write(f"FAILED {filename}: {error}")
    rows.append(row)
    # Save after every item so completed results survive an interrupted run.
    with manifest.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

failed = [r for r in rows if r["status"] == "failed"]
for status in ("downloaded", "skipped", "failed"):
    print(status + ":", sum(r["status"] == status for r in rows))
print("Missing catalogue dekads:", len(missing))
print("Output folder:", OUTPUT_DIR.resolve())
print("Check the Dropbox desktop app for upload/sync completion.")
if failed:
    raise RuntimeError(f"{len(failed)} downloads failed. See {manifest}; fix access and rerun.")